# Customer Churn Prediction

A professional machine-learning workflow for predicting customer churn using a Random Forest classifier.

This notebook improves the original workflow by using train-only preprocessing, median/mode imputation, one-hot encoding, a validation split, class-balanced learning, and final inference on the holdout test set.

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing and modeling
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

# Evaluation
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve
)

RANDOM_STATE = 42


In [ ]:
# Kaggle input paths
train_path = "/kaggle/input/datasets/jpacse/datasets-for-churn-telecom/cell2celltrain.csv"
test_path = "/kaggle/input/datasets/jpacse/datasets-for-churn-telecom/cell2cellholdout.csv"

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

print("Train shape:", train.shape)
print("Test shape:", test.shape)
train.info()


## 1. Initial inspection

The target is `Churn`. The original notebook contains `Yes/No` labels, so they are converted to `1/0`. We keep the test set untouched until inference.

In [ ]:
# Convert the target to binary labels.
train["Churn"] = train["Churn"].map({"Yes": 1, "No": 0})

if train["Churn"].isna().any():
    raise ValueError("Unexpected values were found in the Churn column.")

print(train["Churn"].value_counts())
print("\nChurn ratio:")
print(train["Churn"].value_counts(normalize=True))


In [ ]:
# Inspect missing values before preprocessing.
missing = train.isna().sum().sort_values(ascending=False)
print(missing[missing > 0])


In [ ]:
# Churn distribution
sns.countplot(data=train, x="Churn")
plt.title("Churn Distribution")
plt.xlabel("Churn (0 = No, 1 = Yes)")
plt.ylabel("Number of Customers")
plt.show()


## 2. Prepare features and target

`CustomerID` is an identifier rather than a predictive customer characteristic, so it is excluded from `X`. It is preserved separately so it can be included in the final prediction CSV.

In [ ]:
TARGET = "Churn"
ID_COLUMN = "CustomerID"

# Keep IDs for the final submission/output file.
test_ids = test[ID_COLUMN].copy() if ID_COLUMN in test.columns else pd.Series(test.index, name=ID_COLUMN)

# Separate target from predictors and remove the identifier.
X = train.drop(columns=[TARGET, ID_COLUMN], errors="ignore")
y = train[TARGET]

# Test features must have exactly the same predictor columns as X.
X_test_final = test.drop(columns=[TARGET, ID_COLUMN], errors="ignore")
X_test_final = X_test_final.reindex(columns=X.columns)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Test feature shape:", X_test_final.shape)


## 3. Train / validation split

We reserve 20% of the labeled training data for local validation. `stratify=y` keeps the churn/non-churn proportion similar in both subsets.

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training rows:", X_train.shape[0])
print("Validation rows:", X_valid.shape[0])


## 4. Preprocessing

### Numerical features
Missing numerical values are replaced with the **median calculated from the training split**.

### Categorical features
Missing categorical values are replaced with the **most frequent value**.

### Encoding
`OneHotEncoder(handle_unknown="ignore")` learns categories from the training split only. If a category appears only in validation/test, it will not cause a dimension mismatch.

In [ ]:
numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number"]).columns.tolist()

print("Number of numerical features:", len(numeric_features))
print("Number of categorical features:", len(categorical_features))

# Numerical pipeline: median imputation.
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

# Categorical pipeline: mode imputation + one-hot encoding.
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ],
    remainder="drop"
)


## 5. Random Forest model

The target is moderately imbalanced (`Churn=1` is the minority class). `class_weight="balanced"` gives more weight to the minority class during training.

In [ ]:
model = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

# One pipeline keeps preprocessing and modeling together and prevents leakage.
clf = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", model)
])

clf.fit(X_train, y_train)


## 6. Validation evaluation

We evaluate on the validation set using classification report, confusion matrix, and ROC-AUC. ROC-AUC uses predicted probabilities rather than hard class labels.

In [ ]:
y_valid_pred = clf.predict(X_valid)
y_valid_proba = clf.predict_proba(X_valid)[:, 1]

print("Classification Report")
print(classification_report(y_valid, y_valid_pred, digits=4))

roc_auc = roc_auc_score(y_valid, y_valid_proba)
print(f"ROC-AUC: {roc_auc:.4f}")

cm = confusion_matrix(y_valid, y_valid_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["No Churn", "Churn"],
    yticklabels=["No Churn", "Churn"]
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Validation Confusion Matrix")
plt.tight_layout()
plt.show()

# ROC curve
fpr, tpr, _ = roc_curve(y_valid, y_valid_proba)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"Random Forest (AUC = {roc_auc:.3f})")
plt.plot([0, 1], [0, 1], "--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Validation Set")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## 7. Feature importance

After one-hot encoding, the Random Forest sees encoded feature names. The code below extracts those names and displays the most important encoded features.

In [ ]:
# Get feature names after preprocessing.
feature_names = clf.named_steps["preprocessor"].get_feature_names_out()
importances = clf.named_steps["model"].feature_importances_

importance_df = (
    pd.DataFrame({"Feature": feature_names, "Importance": importances})
    .sort_values("Importance", ascending=False)
    .head(20)
)

plt.figure(figsize=(10, 8))
sns.barplot(data=importance_df, x="Importance", y="Feature")
plt.title("Top 20 Feature Importances")
plt.tight_layout()
plt.show()

importance_df


## 8. Train the final model on all labeled training data

After validation is complete, we fit a fresh pipeline on **all labeled training rows**. This uses the maximum amount of labeled data before making predictions on the holdout test set.

In [ ]:
# Build a fresh preprocessing pipeline using the full training data.
final_numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
final_categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

final_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))]),
            final_numeric_features
        ),
        (
            "cat",
            Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore"))
            ]),
            final_categorical_features
        )
    ],
    remainder="drop"
)

final_model = Pipeline(steps=[
    ("preprocessor", final_preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

final_model.fit(X, y)


## 9. Inference on the holdout test set and CSV export

The test set is transformed by the preprocessing fitted on the full labeled training data. The resulting predictions are saved as `customer_churn_predictions.csv`.

In [ ]:
test_predictions = final_model.predict(X_test_final)

submission = pd.DataFrame({
    ID_COLUMN: test_ids.values,
    TARGET: test_predictions.astype(int)
})

output_path = "customer_churn_predictions.csv"
submission.to_csv(output_path, index=False)

print(f"Saved predictions to: {output_path}")
print(submission.head())
print("Prediction shape:", submission.shape)
print("Predicted churn distribution:")
print(submission[TARGET].value_counts())


## 10. Optional: save churn probabilities

If you want to prioritize customers by risk instead of using only a 0/1 prediction, save the probability of churn as well.

In [ ]:
test_churn_probability = final_model.predict_proba(X_test_final)[:, 1]

submission_with_probability = pd.DataFrame({
    ID_COLUMN: test_ids.values,
    "Churn": test_predictions.astype(int),
    "ChurnProbability": test_churn_probability
})

submission_with_probability.to_csv("customer_churn_predictions_with_probability.csv", index=False)
print(submission_with_probability.head())
